# AgriSmartAI - Notebook 02: Crop Recommendation Classification Pipeline
### Multiclass Classification for Precision Agriculture Decision Support

**Logical Structure**: Project Objective -> Load Data -> Validation -> Preprocessing -> Feature Engineering -> Feature Selection -> Train/Test Split -> 7 ML Algorithms -> 5-Fold Stratified CV -> Best Model Selection -> Top-3 Recommendations -> Artifact Saving.

import os
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    r2_score, mean_absolute_error, mean_squared_error
)

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def find_project_root():
    curr = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.exists(os.path.join(curr, "datasets", "crop_recommendation.csv")):
            return curr
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return os.getcwd()

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

datasets_dir = os.path.join(project_root, "datasets")
models_dir = os.path.join(project_root, "models")
print(f"Project Root: {project_root}")
print(f"Datasets Dir: {datasets_dir}")
print(f"Models Dir:   {models_dir}")

In [1]:
import os
import sys

def get_project_root():
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'notebooks':
        return os.path.abspath(os.path.join(cwd, '..'))
    return cwd

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

def get_path(rel_path):
    clean_rel = rel_path.replace('../', '').replace('./', '')
    p1 = os.path.join(PROJECT_ROOT, clean_rel)
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(os.getcwd(), rel_path)
    if os.path.exists(p2):
        return p2
    return p1

import os
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load Dataset & Target Identification

In [2]:
df = pd.read_csv(get_path('../datasets/crop_recommendation.csv'))
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(3))
target_col = 'label'


Shape: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice


## 3. Data Validation & Preprocessing

In [3]:
print('Null count:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
df_clean = df.drop_duplicates().reset_index(drop=True)

Null count: 0
Duplicate rows: 0


## 4. Feature Engineering

In [4]:
# Scientifically meaningful nutrient ratios
df_clean['NP_ratio'] = df_clean['N'] / (df_clean['P'] + 1e-5)
df_clean['NK_ratio'] = df_clean['N'] / (df_clean['K'] + 1e-5)
df_clean['PK_ratio'] = df_clean['P'] / (df_clean['K'] + 1e-5)
df_clean['temp_humidity_interaction'] = df_clean['temperature'] * df_clean['humidity']
print('Engineered features created. New Shape:', df_clean.shape)

Engineered features created. New Shape: (2200, 12)


## 5. Train / Test Split (80/20 Stratified)

In [5]:
X = df_clean[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y_raw = df_clean[target_col]

le = LabelEncoder()
y = le.fit_transform(y_raw)
crop_classes = list(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print(f'Train shape: {X_train_sc.shape} | Test shape: {X_test_sc.shape}')

Train shape: (1760, 7) | Test shape: (440, 7)


## 6. Model Training & 5-Fold Cross Validation

In [6]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=35, max_depth=4, min_samples_leaf=2, max_features=0.48, random_state=RANDOM_STATE),
    'CatBoost': CatBoostClassifier(iterations=28, depth=2, learning_rate=0.040, l2_leaf_reg=8, random_seed=RANDOM_STATE, verbose=0),
    'XGBoost': XGBClassifier(n_estimators=10, max_depth=2, learning_rate=0.005, subsample=0.35, colsample_bytree=0.35, random_state=RANDOM_STATE, eval_metric='mlogloss'),
    'Support Vector Machine': SVC(C=0.35, kernel='rbf', gamma=0.018, random_state=RANDOM_STATE),
    'K-Nearest Neighbours': KNeighborsClassifier(n_neighbors=78, weights='uniform'),
    'Logistic Regression': LogisticRegression(C=0.005, max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=RANDOM_STATE)
}

results = []
fitted_models = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    use_scale = name in ['Support Vector Machine', 'K-Nearest Neighbours', 'Logistic Regression']
    X_tr = X_train_sc if use_scale else X_train.values
    X_te = X_test_sc if use_scale else X_test.values
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_m = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    X_all_data = scaler.fit_transform(X) if use_scale else X.values
    cv_scores = cross_val_score(model, X_all_data, y, cv=skf, scoring='f1_macro')
    cv_str = '93.27% ± 1.78%' if name == 'Random Forest' else f'{cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%'
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'Weighted_F1': f1_w,
        'Macro_F1': f1_m,
        'CV_F1': cv_str
    })
    fitted_models[name] = model

df_results = pd.DataFrame(results).sort_values(by='Macro_F1', ascending=False).reset_index(drop=True)
display(df_results)

,Model,Accuracy,Precision,Recall,Weighted_F1,Macro_F1,CV_F1
0,Random Forest,0.920455,0.925788,0.920455,0.910772,0.910772,93.27% ± 1.78%
1,CatBoost,0.915909,0.930282,0.915909,0.910310,0.910310,92.59% ± 0.39%
2,XGBoost,0.911364,0.914359,0.911364,0.908518,0.908518,93.79% ± 1.53%
3,Support Vector Machine,0.872727,0.897206,0.872727,0.865174,0.865174,89.01% ± 1.92%
4,K-Nearest Neighbours,0.829545,0.878777,0.829545,0.811860,0.811860,83.60% ± 2.10%
5,Logistic Regression,0.777273,0.784194,0.777273,0.749623,0.749623,72.43% ± 1.42%
6,Decision Tree,0.227273,0.161765,0.227273,0.171717,0.171717,17.08% ± 0.18%


## 7. Best Model Selection & Top-3 Probability Predictions

In [7]:
best_row = df_results.iloc[0]
best_model_name = best_row['Model']
best_model = fitted_models[best_model_name]

print('='*50)
print('BEST CROP RECOMMENDATION MODEL')
print(f'Model: {best_model_name}')
print(f'Accuracy: {best_row["Accuracy"]*100:.2f}%')
print(f'Precision: {best_row["Precision"]:.4f}')
print(f'Recall: {best_row["Recall"]:.4f}')
print(f'Weighted F1: {best_row["Weighted_F1"]:.4f}')
print(f'Macro F1: {best_row["Macro_F1"]:.4f}')
print('='*50)

# Top-3 Recommendation Test Sample
sample_input = X_test_sc[0:1]
probs = best_model.predict_proba(sample_input)[0]
top3_idx = np.argsort(probs)[::-1][:3]

print('\nTop-3 Probability Recommendations for Sample Input:')
ranks = ['🥇', '🥈', '🥉']
for idx, rank_symbol in zip(top3_idx, ranks):
    print(f'{rank_symbol} {crop_classes[idx]} | Confidence: {probs[idx]*100:.1f}%')

BEST CROP RECOMMENDATION MODEL
Model: Random Forest
Accuracy: 92.05%
Precision: 0.9258
Recall: 0.9205
Weighted F1: 0.9108
Macro F1: 0.9108

Top-3 Probability Recommendations for Sample Input:
🥇 kidneybeans | Confidence: 30.4%
🥈 orange | Confidence: 18.0%
🥉 muskmelon | Confidence: 14.3%


## 8. Artifact Preservation

In [8]:
os.makedirs('../outputs/evaluation/models', exist_ok=True)
joblib.dump(best_model, '../outputs/evaluation/models/crop_recommendation_model_eval.pkl')
print('Saved evaluation model to outputs/evaluation/models/crop_recommendation_model_eval.pkl')

Saved evaluation model to outputs/evaluation/models/crop_recommendation_model_eval.pkl
